In [ ]:
import json, re, textwrap, datetime, base64, os
from collections import defaultdict, Counter
import requests
print('✅  Imports complete')


✅  Imports complete


## Download EMB3D Dataset & Build Index

Downloads the STIX 2.1 bundle from MITRE GitHub and builds all lookup structures.


In [ ]:
EMB3D_URL = 'https://raw.githubusercontent.com/mitre/emb3d/refs/heads/main/assets/emb3d-stix-2.0.1.json'

print(f'Downloading EMB3D STIX dataset...')
print(f'Source: {EMB3D_URL}')

resp    = requests.get(EMB3D_URL, timeout=30)
resp.raise_for_status()
bundle  = resp.json()
objects = bundle['objects']

# ── Partition by STIX type ────────────────────────────────────────────────────
properties_list  = [o for o in objects if o['type'] == 'x-mitre-emb3d-property']
threats_list     = [o for o in objects if o['type'] == 'vulnerability']
mitigations_list = [o for o in objects if o['type'] == 'course-of-action']
relationships    = [o for o in objects if o['type'] == 'relationship']

# ── Lookup dicts ─────────────────────────────────────────────────────────────
prop_by_id    = {o['id']: o for o in properties_list}
prop_by_pid   = {o['x_mitre_emb3d_property_id']: o for o in properties_list}
threat_by_id  = {o['id']: o for o in threats_list}
coa_by_id     = {o['id']: o for o in mitigations_list}

# ── Relationship graphs ───────────────────────────────────────────────────────
prop_to_threat_ids = defaultdict(list)   # property_stix_id -> [threat_stix_ids]
threat_to_coa_ids  = defaultdict(list)   # threat_stix_id   -> [coa_stix_ids]

for rel in relationships:
    rt, src, tgt = rel['relationship_type'], rel['source_ref'], rel['target_ref']
    if rt == 'relates-to':
        if   src in prop_by_id and tgt in threat_by_id: prop_to_threat_ids[src].append(tgt)
        elif tgt in prop_by_id and src in threat_by_id: prop_to_threat_ids[tgt].append(src)
    elif rt == 'mitigates':
        if src in coa_by_id and tgt in threat_by_id: threat_to_coa_ids[tgt].append(src)

# ── Build the property catalog string (passed to Claude) ─────────────────────
PROPERTY_CATALOG = '\n'.join(
    f"{p['x_mitre_emb3d_property_id']} | {p['category']}"
    f" | sub={p['is_subproperty']} | {p['name']}"
    for p in sorted(properties_list, key=lambda x: x['x_mitre_emb3d_property_id'])
)

print(f'\n✅  Dataset loaded and indexed')
print(f'   Properties : {len(properties_list)}')
print(f'   Threats    : {len(threats_list)}')
print(f'   Mitigations: {len(mitigations_list)}')
print(f'   Relationships: {len(relationships)}')


Source: https://raw.githubusercontent.com/mitre/emb3d/refs/heads/main/assets/emb3d-stix-2.0.1.json

✅  Dataset loaded and indexed
   Properties : 59
   Threats    : 81
   Mitigations: 89
   Relationships: 343


## Select a Product for Threat Modeling (PART I)

Choose a product from the list below to dynamically load its system definition and block diagrams for threat modeling.

In [ ]:
import os
import gradio as gr

# Clone the repository locally
REPO_DIR = 'Products4SecurityAnalysis_repo'
print(f'Cloning repository into {REPO_DIR}...')

# Check if the directory already exists to prevent re-cloning errors
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/TheCyberAI/Products4SecurityAnalysis.git $REPO_DIR
else:
    print(f"Repository '{REPO_DIR}' already exists. Skipping clone.")

# --- Collect product folder names ---
PRODUCT_FOLDER_OPTIONS = []

if os.path.exists(REPO_DIR) and os.path.isdir(REPO_DIR):
    print(f'\nAll products found in {REPO_DIR}:')
    for item in os.listdir(REPO_DIR):
        if item == '.git': # Skip the .git folder
            continue
        item_path = os.path.join(REPO_DIR, item)
        if os.path.isdir(item_path):
            PRODUCT_FOLDER_OPTIONS.append(item)
            print(f'- {item}')
    PRODUCT_FOLDER_OPTIONS.sort() # Sort for consistent display
else:
    print(f'Error: Repository directory not found at {REPO_DIR}. Please ensure the repository was cloned correctly.')

# --- Gradio UI for selection ---
# Global variable to store the selected folder
global SELECTED_PRODUCT_FOLDER
SELECTED_PRODUCT_FOLDER = None

def select_product_folder(folder_name):
    global SELECTED_PRODUCT_FOLDER
    SELECTED_PRODUCT_FOLDER = folder_name
    return f"Selected product folder: **{folder_name}**"

if PRODUCT_FOLDER_OPTIONS:
    with gr.Blocks(theme=gr.themes.Soft()) as product_selector_ui:
        gr.Markdown("## Select a Product for Threat Modeling")
        gr.Markdown("Choose a product from the list below. This selection will update the `SELECTED_PRODUCT_FOLDER` variable for subsequent operations.")

        dropdown = gr.Dropdown(
            choices=PRODUCT_FOLDER_OPTIONS,
            label="Product Folders",
            value=PRODUCT_FOLDER_OPTIONS[0] if PRODUCT_FOLDER_OPTIONS else None, # Pre-select first item if available
            interactive=True
        )
        output_text = gr.Markdown(value="Select a product to begin.")

        dropdown.change(
            fn=select_product_folder,
            inputs=[dropdown],
            outputs=[output_text]
        )

    print("\nLaunching Gradio UI for product selection...")
    product_selector_ui.launch(share=True)

    # Set initial SELECTED_PRODUCT_FOLDER after launching the UI
    # This assumes the user will make a selection or the first item is a reasonable default.
    if PRODUCT_FOLDER_OPTIONS and SELECTED_PRODUCT_FOLDER is None:
        SELECTED_PRODUCT_FOLDER = PRODUCT_FOLDER_OPTIONS[0]
        print(f"Initial product selection (if not changed in UI): {SELECTED_PRODUCT_FOLDER}")
else:
    print("No product folders found to create a selection UI.")


Cloning repository into Products4SecurityAnalysis_repo...
Cloning into 'Products4SecurityAnalysis_repo'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 33 (delta 8), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 3.41 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (8/8), done.

All products found in Products4SecurityAnalysis_repo:
- Medical
- Smart Industrial IoT Controller
- Missiles


/tmp/ipykernel_797/200920970.py:41: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as product_selector_ui:



Launching Gradio UI for product selection...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e71e0c85b9afcd2e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Initial product selection (if not changed in UI): Medical


## Prepare ITEM_DEFINITIONS for selected product
Loads the selected product discriptions and prepares the ITEM_DEFINITIONS to be fed to AI model.

**`ITEM_DEFINITIONS`** should list every component with its relevant attributes —
the more detail you provide, the more accurate the property matching will be.


In [ ]:
import json
import os # Added for path manipulation

# Construct the path to the itemDefinition.json file
# REPO_DIR and SELECTED_PRODUCT_FOLDER are assumed to be defined in previous cells
system_definition_path = os.path.join(REPO_DIR, SELECTED_PRODUCT_FOLDER, 'ItemDefination.json')

print(f'Loading item definition from: {system_definition_path}')

try:
    with open(system_definition_path, 'r') as f:
        system_json = json.load(f)

    SYSTEM_NAME        = system_json['system_name']
    SYSTEM_DESCRIPTION = system_json['system_description']

    components_definitions_list = []
    for comp_name, comp_details in system_json['components'].items():
        detail_strings = []
        for key, value in comp_details.items():
            if isinstance(value, dict):
                # Handle nested dictionaries: e.g., protocols: (Modbus TCP: ..., MQTT: ...)
                sub_details = []
                for sk, sv in value.items():
                    if isinstance(sv, dict):
                        # Handle even deeper nesting (e.g., 'port': 502 within 'Modbus TCP')
                        deep_sub_details = ', '.join([f"{dsk}: {dsv}" for dsk, dsv in sv.items()])
                        sub_details.append(f"{sk}: {{{deep_sub_details}}}")
                    else:
                        sub_details.append(f"{sk}: {sv}")
                detail_strings.append(f"{key}: ({'; '.join(sub_details)})") # Use semicolon for inner separation
            elif isinstance(value, list):
                # Handle lists: e.g., interfaces: [Ethernet]
                detail_strings.append(f"{key}: [{'; '.join(map(str, value))}]")
            else:
                # Basic key-value pair
                detail_strings.append(f"{key}: {value}")

        # Format the component name and join its details
        formatted_comp_name = comp_name.replace('_', ' ').title()
        components_definitions_list.append(f"{formatted_comp_name:<9}: {', '.join(detail_strings)}")

    # Combine SYSTEM_NAME, SYSTEM_DESCRIPTION, and components_definitions_list for ITEM_DEFINITIONS
    ITEM_DEFINITIONS = f"SYSTEM NAME: {SYSTEM_NAME}\n\nSYSTEM DESCRIPTION:\n{SYSTEM_DESCRIPTION}\n\nCOMPONENTS:\n" + '\n'.join(components_definitions_list)

    #print(f'✅  System definition loaded for: {SYSTEM_NAME}')
    #print('\n--- SYSTEM DESCRIPTION ---\n')
    #print(SYSTEM_DESCRIPTION)
    print('\n--- ITEM DEFINITIONS ---\n')
    print(ITEM_DEFINITIONS)

except FileNotFoundError:
    print(f'Error: itemDefinition.json not found at {system_definition_path}')
    print('Please ensure the file exists in the selected product folder.')
except json.JSONDecodeError as e:
    print(f'Error parsing itemDefinition.json: {e}')
    print('Please ensure the file is a valid JSON.')
except KeyError as e:
    print(f'Error parsing system definition: Missing key {e}')
    print('Please ensure the JSON file has system_name, system_description, and components.')
except Exception as e:
    print(f'An unexpected error occurred: {e}')

Loading item definition from: Products4SecurityAnalysis_repo/Smart Industrial IoT Controller/ItemDefination.json

--- ITEM DEFINITIONS ---

SYSTEM NAME: Smart Industrial IoT Controller

SYSTEM DESCRIPTION:
An industrial IoT controller built around an ARM Cortex-A53 quad-core SoC running Linux kernel 5.10 with 512 MB DDR4 RAM and 4 GB eMMC flash storage. A JTAG debug port and UART serial console are physically accessible on the PCB with no protection or fusing. The device communicates over Ethernet using Modbus TCP on port 502 and MQTT on port 1883 with no TLS. An HTTP web management interface runs on port 80 with no authentication required by default. Firmware updates are delivered over-the-air via MQTT but are neither signed nor encrypted. U-Boot bootloader is present with no secure boot chain configured. No TPM or hardware root of trust is present. The application is written in C and C++.

COMPONENTS:
Cpu      : type: ARM Cortex-A53, cores: 4, frequency: 1.2 GHz, secure_boot: False
R

## Load Hardware bloack diagram and Software bloack Diagram
Loads  the Hardware bloack diagram and Software bloack Diagram for selected product

In [ ]:
# ── Upload block diagram ──────────────────────────────────────────────────────
import os

HARDWARE_DIAGRAM_BYTES = None
HARDWARE_DIAGRAM_MIME  = None
SOFTWARE_DIAGRAM_BYTES = None
SOFTWARE_DIAGRAM_MIME  = None

# List of common image extensions to check
IMAGE_EXTENSIONS = ['png', 'jpg', 'jpeg', 'gif', 'webp']

# Function to find diagram file with any supported extension
def find_diagram_file(base_name, product_folder, repo_dir):
    for ext in IMAGE_EXTENSIONS:
        file_name = f"{base_name}.{ext}"
        full_path = os.path.join(repo_dir, product_folder, file_name)
        if os.path.exists(full_path):
            return full_path
    return None

# Determine MIME type based on file extension
def get_mime_type(file_path):
    if file_path is None:
        return None
    ext = file_path.rsplit('.', 1)[-1].lower()
    return {
        'png': 'image/png',
        'jpg': 'image/jpeg',
        'jpeg': 'image/jpeg',
        'gif': 'image/gif',
        'webp': 'image/webp'
    }.get(ext, 'application/octet-stream') # Default to octet-stream if unknown

# --- Load Hardware Diagram ---
hardware_diagram_path = find_diagram_file('Hardware_block_diagram', SELECTED_PRODUCT_FOLDER, REPO_DIR)
print(f'Attempting to load hardware block diagram from: {hardware_diagram_path if hardware_diagram_path else 'None (file not found)'}')
try:
    if hardware_diagram_path:
        with open(hardware_diagram_path, 'rb') as f:
            HARDWARE_DIAGRAM_BYTES = f.read()
        HARDWARE_DIAGRAM_MIME = get_mime_type(hardware_diagram_path)
        print(f'✅  Hardware Diagram loaded: {os.path.basename(hardware_diagram_path)}  ({len(HARDWARE_DIAGRAM_BYTES):,} bytes)')
    else:
        print('Error: Hardware block diagram not found with supported extensions.')
        print('No hardware diagram loaded.')
except Exception as e:
    print(f'An unexpected error occurred loading hardware diagram: {e}')
    print('No hardware diagram loaded.')

# --- Load Software Diagram ---
software_diagram_path = find_diagram_file('Software_block_diagram', SELECTED_PRODUCT_FOLDER, REPO_DIR)
print(f'\nAttempting to load software block diagram from: {software_diagram_path if software_diagram_path else 'None (file not found)'}')
try:
    if software_diagram_path:
        with open(software_diagram_path, 'rb') as f:
            SOFTWARE_DIAGRAM_BYTES = f.read()
        SOFTWARE_DIAGRAM_MIME = get_mime_type(software_diagram_path)
        print(f'✅  Software Diagram loaded: {os.path.basename(software_diagram_path)}  ({len(SOFTWARE_DIAGRAM_BYTES):,} bytes)')
    else:
        print('Error: Software block diagram not found with supported extensions.')
        print('No software diagram loaded.')
except Exception as e:
    print(f'An unexpected error occurred loading software diagram: {e}')
    print('No software diagram loaded.')

Attempting to load hardware block diagram from: Products4SecurityAnalysis_repo/Smart Industrial IoT Controller/Hardware_block_diagram.png
✅  Hardware Diagram loaded: Hardware_block_diagram.png  (1,644,461 bytes)

Attempting to load software block diagram from: Products4SecurityAnalysis_repo/Smart Industrial IoT Controller/Software_block_diagram.png
✅  Software Diagram loaded: Software_block_diagram.png  (1,791,388 bytes)


## AI-Powered Threat Analysis

Three steps, all driven by Claude:

1. **Property identification** — AI reads the full property EMB3D catalog and
   your system input (Hardware bloack diagram and Software bloack Diagram + item definitions) and
   returns every applicable PID with a justification
2. **STIX graph resolution** — walks `PID → TID → MID` deterministically
   using the downloaded STIX data
3. **Security report** — AI synthesises a Report


In [ ]:
import json
import re
from collections import defaultdict

# IMPORT THE NEW MODERN SDK
from google import genai
from google.genai import types as genai_types
from google.colab import userdata, files

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# Initialize the modern client
client = genai.Client(api_key=GEMINI_API_KEY)

# Define the model to be used
#MODEL = "gemini-flash" # Assuming a multimodal model suitable for images and text
MODEL = "gemini-2.5-flash"

# ── Build the prompt ─────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    'You are an embedded systems security expert. '
    'You will receive system block diagram,hardware block diagram (images) and item definitions (text), '
    'together with the complete MITRE EMB3D property catalog. '
    'Your task is to identify every property from the catalog that applies to this device. '
    'A property applies if the device has that component, capability, or characteristic — '
    'including implicit ones (e.g. a Linux device running C code has PID-3122). '
    'Include both parent properties and relevant sub-properties. '
    'Return a JSON array where each element follows this schema: '
    '{"pid": "PID-XX", "name": "<property name>", "justification": "one sentence"}.'
)

USER_TEXT = (
    f'ITEM DEFINITIONS:\n{ITEM_DEFINITIONS.strip()}\n\n'
    f'EMB3D PROPERTY CATALOG:\n{PROPERTY_CATALOG}\n\n'
    'Identify every applicable property. Return a JSON array.'
)

# ── Assemble content parts ────────────────────────────────────────────────────
parts = []

# Add Hardware Diagram if available
if HARDWARE_DIAGRAM_BYTES and HARDWARE_DIAGRAM_MIME:
    parts.append(genai_types.Part.from_bytes(data=HARDWARE_DIAGRAM_BYTES, mime_type=HARDWARE_DIAGRAM_MIME))
    parts.append(genai_types.Part.from_text(
        text='Hardware system block diagram above.'))

# Add Software Diagram if available
if SOFTWARE_DIAGRAM_BYTES and SOFTWARE_DIAGRAM_MIME:
    parts.append(genai_types.Part.from_bytes(data=SOFTWARE_DIAGRAM_BYTES, mime_type=SOFTWARE_DIAGRAM_MIME))
    parts.append(genai_types.Part.from_text(
        text='Software system block diagram above.'))

# Add a combined descriptive text if any diagrams were added
if HARDWARE_DIAGRAM_BYTES or SOFTWARE_DIAGRAM_BYTES:
    parts.append(genai_types.Part.from_text(
        text='System block diagrams above. Item definitions and property catalog below.'))


parts.append(genai_types.Part.from_text(text=USER_TEXT))

# ── Call Gemini ───────────────────────────────────────────────────────────────
print(f'Sending to {MODEL}...')
response = client.models.generate_content(
    model    = MODEL,
    contents = parts,
    config   = genai_types.GenerateContentConfig(
        system_instruction = SYSTEM_PROMPT,
        max_output_tokens  = 8192,
        # Instructs the model structure to strictly return valid JSON
        response_mime_type = "application/json"
    ),
)

# ── Parse response ────────────────────────────────────────────────────────────
# Because we used response_mime_type, response.text will be pure JSON
# without markdown ```json wrappers, but we keep the strip just in case.
raw  = response.text.strip()

# Fix: Remove null bytes, which cause SyntaxError in json.loads
raw_cleaned = raw.replace('\x00', '')

# Fix: Filter out lines containing non-ASCII characters that are not valid JSON structure
# This addresses cases where the LLM might inject extraneous text (e.g., 'を行い、')
# that breaks JSON parsing, while preserving legitimate unicode within string values.
lines = raw_cleaned.split('\n') # Use raw_cleaned here to apply previous step's result
cleaned_lines = []
for line in lines:
    # Heuristic: if a line contains non-ASCII characters and doesn't start with typical JSON structure elements
    # (quotes, braces, brackets), it's likely extraneous chatter and should be skipped.
    if re.search(r'[^\u0000-\u007F]', line) and not (
        line.strip().startswith('"') or
        line.strip().startswith('{') or
        line.strip().startswith('[') or
        line.strip().startswith(']') or
        line.strip().startswith('}') or
        line.strip().replace(':', '', 1).strip().startswith('"') # Check for key-value pair where key is quoted
    ):
        continue
    cleaned_lines.append(line)
raw_cleaned = '\n'.join(cleaned_lines)

# Fix: Replace common malformed JSON pattern from LLM output (e.g., '} - {' with '},{')
# This is kept in addition to the new cleaning step as it handles a different type of error.
raw_cleaned = re.sub(r'}\s*-\s*{', '},{', raw_cleaned)
data = json.loads(raw_cleaned)

# ── Display results ───────────────────────────────────────────────────────────
print(f'\n{len(data)} properties identified\n')
print(f'{"PID":<12} {"Category":<22} {"Property Name"}')
print('-' * 90)

# Group by category for readability
by_cat = defaultdict(list)
for item in data:
    pid  = item.get('pid', '')
    prop = next((p for p in properties_list
                 if p['x_mitre_emb3d_property_id'] == pid), None)
    cat  = prop['category'] if prop else 'Unknown'
    by_cat[cat].append(item)

for cat in ['Hardware', 'System Software', 'Application Software', 'Networking']:
    items = by_cat.get(cat, [])
    if not items:
        continue
    print(f'\n[{cat}]')
    for item in sorted(items, key=lambda x: x.get('pid','')):
        pid   = item.get('pid', '')
        name  = item.get('name', '')
        just  = item.get('justification', '')
        prop  = next((p for p in properties_list
                      if p['x_mitre_emb3d_property_id'] == pid), None)
        sub   = '  (sub)' if prop and prop.get('is_subproperty') else ''
        print(f'  {pid:<12} {name}{sub}')
        print(f'               → {just}')


Sending to gemini-2.5-flash...

35 properties identified

PID          Category               Property Name
------------------------------------------------------------------------------------------

[Hardware]
  PID-11       Device includes a microprocessor
               → The device includes an ARM Cortex-A53 quad-core SoC.
  PID-12       Device includes Memory/Storage (external to CPU)
               → The device includes 512 MB DDR4 RAM and 4 GB eMMC flash storage, both external to the SoC.
  PID-121      Device includes buses for external memory/storage  (sub)
               → The block diagram shows a RAM Bus for DDR4 RAM and an eMMC Interface for eMMC flash.
  PID-122      Device includes discrete chips/devices that have access to the same physical memory  (sub)
               → The ARM SoC, Memory Controller, and eMMC Controller interact with the shared DDR4 RAM and eMMC Flash.
  PID-123      Device includes ROM, VRAM, or removable Storage  (sub)
               → The device in

### Listing Available Models

Let's check which models are actually available with your current API key and support the `generateContent` method. This will help us choose a valid model identifier.

In [ ]:
import pandas as pd
import inspect

models = client.models.list()

available_models = []
# Debugging: Print attributes of the first model object to understand its structure
if models:
    first_model = next(iter(models))
    print(f"Debugging Model Object: {first_model.name if hasattr(first_model, 'name') else first_model}")
    print(f"Available attributes: {dir(first_model)}")
    # Add printing the content of supported_actions
    if hasattr(first_model, 'supported_actions'):
        print(f"Content of supported_actions: {first_model.supported_actions}")
    print("--- End Debugging Info ---")

for m in models:
    # Attempt to use the officially documented attribute, or a discovered one like 'supported_actions'
    if hasattr(m, 'supported_actions'):
        # Iterate through the supported actions and check for 'generateContent'
        # The 'supported_actions' attribute is a list of dictionaries, so we need to iterate.
        supports_generate_content = any('method' in action and action['method'] == 'generateContent' for action in m.supported_actions)

        if supports_generate_content:
            available_models.append({
                'name': m.name,
                'display_name': m.display_name,
                'version': m.version,
                'description': m.description,
                'supported_methods': [action['method'] for action in m.supported_actions if 'method' in action] # Store all supported methods for this model
            })
    # If the standard attribute is not found, it implies a problem, or a different object type
    else:
        # This case should ideally not be hit if the library is working as expected.
        # It means `m` does not have `supported_actions` (or the previously tried 'supported_generation_methods').
        # We already printed dir(m) for the first one, so we just skip or log.
        print(f"Skipping model {m.name if hasattr(m, 'name') else 'unknown model'} due to missing 'supported_actions'.")


if not available_models and models:
    print("\nNo models found that support 'generateContent' using 'supported_actions' attribute.")
    print("Please check the debugging information above for available attributes on model objects.")

df_available_models = pd.DataFrame(available_models)
display(df_available_models)

Debugging Model Object: models/gemini-2.5-flash
Available attributes: ['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__', '_

""


From the list above, please identify a model that is suitable for multimodal input (if your task requires images) and supports `generateContent`. Once you have chosen a model name (from the 'name' column, e.g., 'gemini-1.5-pro-001'), let me know, and I will update the `MODEL` variable in cell `Tkh83-W6UGKy`.

In [ ]:
import json
import re
from collections import defaultdict

def analyze_system_for_emb3d_threats(
    item_definitions,
    property_catalog,
    hardware_diagram_bytes=None,
    hardware_diagram_mime=None,
    software_diagram_bytes=None,
    software_diagram_mime=None
):
    """
    Analyzes system definitions and diagrams using a Generative AI model (Gemini)
    to identify EMB3D properties, threats, and mitigations.

    Args:
        item_definitions (str): A string containing the system's item definitions.
        property_catalog (str): A string containing the EMB3D property catalog.
        hardware_diagram_bytes (bytes, optional): Bytes of the hardware diagram image.
        hardware_diagram_mime (str, optional): MIME type of the hardware diagram image.
        software_diagram_bytes (bytes, optional): Bytes of the software diagram image.
        software_diagram_mime (str, optional): MIME type of the software diagram image.

    Returns:
        dict: A dictionary of identified properties, their associated threats, and mitigations.
    """

    # --- Build the prompt ─────────────────────────────────────────────────────────
    SYSTEM_PROMPT = (
        'You are an embedded systems security expert. '
        'You will receive system block diagrams (images) and item definitions (text), '
        'together with the complete MITRE EMB3D property catalog. '
        'Your task is to identify every property from the catalog that applies to this device. '
        'A property applies if the device has that component, capability, or characteristic — '
        'including implicit ones (e.g. a Linux device running C code has PID-3122). '
        'Include both parent properties and relevant sub-properties. '
        'Return a JSON array where each element follows this schema: '
        '{"pid": "PID-XX", "name": "<property name>", "justification": "one sentence"}.'
    )

    USER_TEXT = (
        f'ITEM DEFINITIONS:\n{item_definitions.strip()}\n\n'
        f'EMB3D PROPERTY CATALOG:\n{property_catalog}\n\n'
        'Identify every applicable property. Return a JSON array.'
    )

    # --- Assemble content parts ────────────────────────────────────────────────────
    parts = []

    # Add Hardware Diagram if available
    if hardware_diagram_bytes and hardware_diagram_mime:
        parts.append(genai_types.Part.from_bytes(data=hardware_diagram_bytes, mime_type=hardware_diagram_mime))
        parts.append(genai_types.Part.from_text(text='Hardware system block diagram above.'))

    # Add Software Diagram if available
    if software_diagram_bytes and software_diagram_mime:
        parts.append(genai_types.Part.from_bytes(data=software_diagram_bytes, mime_type=software_diagram_mime))
        parts.append(genai_types.Part.from_text(text='Software system block diagram above.'))

    # Add a combined descriptive text if any diagrams were added
    if hardware_diagram_bytes or software_diagram_bytes:
        parts.append(genai_types.Part.from_text(text='System block diagrams above. Item definitions and property catalog below.'))

    parts.append(genai_types.Part.from_text(text=USER_TEXT))

    # --- Call Gemini ───────────────────────────────────────────────────────────────
    print(f'Sending to {MODEL}...')
    response = client.models.generate_content(
        model    = MODEL,
        contents = parts,
        config   = genai_types.GenerateContentConfig(
            system_instruction = SYSTEM_PROMPT,
            max_output_tokens  = 8192,
            response_mime_type = "application/json"
        ),
    )

    # --- Parse response ────────────────────────────────────────────────────────────
    raw  = response.text.strip()
    data = json.loads(raw)

    # --- Map properties to threats and mitigations ─────────────────────────────────
    identified_threats_mitigations = defaultdict(lambda: defaultdict(list))

    for item in data:
        pid_code = item.get('pid', '')

        prop_stix_id = None
        if pid_code in prop_by_pid:
            prop_stix_id = prop_by_pid[pid_code]['id']

        if prop_stix_id and prop_stix_id in prop_to_threat_ids:
            threat_stix_ids = prop_to_threat_ids[prop_stix_id]

            for threat_id in threat_stix_ids:
                threat_details = threat_by_id.get(threat_id)
                if threat_details:
                    threat_name = threat_details.get('name', 'Unknown Threat')
                    threat_desc = threat_details.get('description', 'No description available.')

                    mitigations_for_threat = []
                    if threat_id in threat_to_coa_ids:
                        coa_stix_ids = threat_to_coa_ids[threat_id]
                        for coa_id in coa_stix_ids:
                            coa_details = coa_by_id.get(coa_id)
                            if coa_details:
                                mitigations_for_threat.append({
                                    'name': coa_details.get('name', 'Unknown Mitigation'),
                                    'description': coa_details.get('description', 'No description available.')
                                })

                    identified_threats_mitigations[pid_code][threat_name].append({
                        'threat_description': threat_desc,
                        'mitigations': mitigations_for_threat
                    })
    return identified_threats_mitigations

## Identified Threats and Mitigations

In [ ]:
# Initialize a dictionary to store identified properties, their threats, and mitigations
identified_threats_mitigations = defaultdict(lambda: defaultdict(list))

# Loop through each identified property (PID)
for item in data:
    pid_code = item.get('pid', '')

    # Get the STIX ID for the identified property
    prop_stix_id = None
    if pid_code in prop_by_pid:
        prop_stix_id = prop_by_pid[pid_code]['id']

    if prop_stix_id and prop_stix_id in prop_to_threat_ids:
        # Get all threat STIX IDs associated with this property
        threat_stix_ids = prop_to_threat_ids[prop_stix_id]

        for threat_id in threat_stix_ids:
            threat_details = threat_by_id.get(threat_id)
            if threat_details:
                threat_name = threat_details.get('name', 'Unknown Threat')
                threat_desc = threat_details.get('description', 'No description available.')

                mitigations_for_threat = []
                if threat_id in threat_to_coa_ids:
                    coa_stix_ids = threat_to_coa_ids[threat_id]
                    for coa_id in coa_stix_ids:
                        coa_details = coa_by_id.get(coa_id)
                        if coa_details:
                            mitigations_for_threat.append({
                                'name': coa_details.get('name', 'Unknown Mitigation'),
                                'description': coa_details.get('description', 'No description available.')
                            })

                identified_threats_mitigations[pid_code][threat_name].append({
                    'threat_description': threat_desc,
                    'mitigations': mitigations_for_threat
                })


# Display the results
if identified_threats_mitigations:
    print('--- Associated Threats and Mitigations ---\n')
    for pid_code, threats_dict in identified_threats_mitigations.items():
        prop_name = next((p['name'] for p in properties_list if p['x_mitre_emb3d_property_id'] == pid_code), pid_code)
        print(f'**Property: {pid_code} - {prop_name}**\n')
        for threat_name, threat_info_list in threats_dict.items():
            print(f'  Threat: {threat_name}')
            for threat_info in threat_info_list:
                print(f'    Description: {textwrap.fill(threat_info['threat_description'], width=90, initial_indent='      ', subsequent_indent='      ')}')
                if threat_info['mitigations']:
                    print('    Mitigations:')
                    for mit in threat_info['mitigations']:
                        print(f'      - {mit['name']}')
                        print(f'        Description: {textwrap.fill(mit['description'], width=90, initial_indent='          ', subsequent_indent='          ')}')
                else:
                    print('    No specific mitigations found for this threat.')
            print('\n')
else:
    print('No threats and mitigations identified for the selected properties.')

--- Associated Threats and Mitigations ---

**Property: PID-11 - Device includes a microprocessor**

  Threat: Power Consumption Analysis Side Channel
    Description:       Devices will oftentimes consume variable amounts of power depending on the
      operations the device is performing. Power consumption analysis involves the reading
      and analyzing of power usage of a device.  If a device is vulnerable to a power
      consumption analysis attack, it may be possible to extract or deduce information
      about the operating state of the device. This can include extracting secrets/keys,
      discovering operations conducted on sections of memory, and device control flow. A
      threat actor can therefore physically monitor the power consumption of a device
      during an execution of a cryptographic operation to create a trace of its power
      usage over time. By leveraging the understanding of the operations of common
      cryptographic properties, the power usage traces

In [ ]:
pip install gradio

In [ ]:
import gradio as gr

def get_property_details(pid_code):
    report_detail_str = ""
    if pid_code in identified_threats_mitigations:
        prop_name_full = next((p['name'] for p in properties_list if p['x_mitre_emb3d_property_id'] == pid_code), pid_code)

        # Get the original justification for the property from the 'data' (Gemini output)
        prop_obj_from_gemini = next((item for item in data if item.get('pid') == pid_code), None)
        prop_justification = prop_obj_from_gemini.get('justification', 'No specific justification provided by Gemini.') if prop_obj_from_gemini else 'No specific justification found.'

        report_detail_str += f"## Property: {pid_code} - {prop_name_full}\n"
        report_detail_str += f"**Justification:** {textwrap.fill(prop_justification, width=90, subsequent_indent=' ')}\n\n"

        threats_dict = identified_threats_mitigations[pid_code]
        if threats_dict:
            report_detail_str += "### Associated Threats:\n\n"
            for threat_name, threat_info_list in threats_dict.items():
                # Use HTML <details><summary> for collapsible sections in Gradio Markdown
                report_detail_str += f"<details><summary><b>Threat: {threat_name}</b></summary>\n"
                for threat_info in threat_info_list:
                    report_detail_str += f"**Description:** {textwrap.fill(threat_info['threat_description'], width=90, subsequent_indent='  ')}\n\n"
                    if threat_info['mitigations']:
                        report_detail_str += "**Mitigations:**\n"
                        for mit in threat_info['mitigations']:
                            report_detail_str += f"- **{mit['name']}**\n"
                            report_detail_str += f"  Description: {textwrap.fill(mit['description'], width=90, subsequent_indent='    ')}\n\n"
                    else:
                        report_detail_str += "  *No specific mitigations found for this threat.*\n\n"
                report_detail_str += "</details>\n\n" # Close details tag
        else:
            report_detail_str += "*No threats identified for this property.*"
    else:
        report_detail_str = f"Please select a property from the dropdown to view its details."
    return report_detail_str

def get_summary():
    total_properties = len(data)
    total_unique_threats = set()
    total_unique_mitigations = set()

    for pid_code, threats_dict in identified_threats_mitigations.items():
        for threat_name, threat_info_list in threats_dict.items():
            for threat_info in threat_info_list:
                total_unique_threats.add(threat_name)
                for mit in threat_info['mitigations']:
                    total_unique_mitigations.add(mit['name'])

    summary_str = f"**Total Identified Properties:** {total_properties}\n"
    summary_str += f"**Total Unique Threats:** {len(total_unique_threats)}\n"
    summary_str += f"**Total Unique Mitigations:** {len(total_unique_mitigations)}\n\n"
    summary_str += "Select a property from the dropdown below to view its associated threats and mitigations."
    return summary_str

# Prepare dropdown choices
property_choices = []
for item in data:
    pid = item.get('pid', '')
    name = item.get('name', '')
    full_prop_name = next((p['name'] for p in properties_list if p['x_mitre_emb3d_property_id'] == pid), name)
    property_choices.append((f"{pid} - {full_prop_name}", pid)) # (display_text, value)
property_choices = sorted(property_choices, key=lambda x: x[0])


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🕵️‍♂️ EMB3D Threat Modeling Report
        ### Interactive Dashboard for Identified Threats and Mitigations
        """
    )

    gr.Markdown(get_summary())

    with gr.Row():
        with gr.Column(scale=1):
            property_dropdown = gr.Dropdown(
                choices=property_choices,
                label="Select an EMB3D Property",
                value=None, # Initially no selection
                interactive=True
            )
        with gr.Column(scale=3):
            output_markdown = gr.Markdown(
                value="Select a property from the dropdown to view its details.",
                label="Property Details, Threats, and Mitigations"
            )

    property_dropdown.change(
        fn=get_property_details,
        inputs=[property_dropdown],
        outputs=[output_markdown]
    )

demo.launch(share=True)

/tmp/ipykernel_797/2669051883.py:65: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aedbbfece3f5a7cc02.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


##Code Analysis (PART II)

In [ ]:
import os

SAMPLE_REPO_DIR = 'SampleItems4SecurityAnalysis_repo'

print(f'Cloning repository into {SAMPLE_REPO_DIR}...')

# Check if the directory already exists to prevent re-cloning errors
if not os.path.exists(SAMPLE_REPO_DIR):
    !git clone https://github.com/TheCyberAI/SampleItems4SecurityAnalysis.git {SAMPLE_REPO_DIR}
    print(f"Repository '{SAMPLE_REPO_DIR}' cloned successfully.")
else:
    print(f"Repository '{SAMPLE_REPO_DIR}' already exists. Skipping clone.")

Cloning repository into SampleItems4SecurityAnalysis_repo...
Cloning into 'SampleItems4SecurityAnalysis_repo'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 28 (delta 1), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 10.75 KiB | 10.75 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Repository 'SampleItems4SecurityAnalysis_repo' cloned successfully.


In [ ]:
import gradio as gr
import os
import json # Added for json parsing/dumping
import re   # Added for regex parsing combined_file_content
from collections import defaultdict
from google import genai
from google.genai import types as genai_types
import textwrap # Added for text formatting in output

# --- Collect folder names from the sample repository ---
SAMPLE_FOLDER_OPTIONS = []

if os.path.exists(SAMPLE_REPO_DIR) and os.path.isdir(SAMPLE_REPO_DIR):
    for item in os.listdir(SAMPLE_REPO_DIR):
        if item == '.git': # Skip the .git folder
            continue
        item_path = os.path.join(SAMPLE_REPO_DIR, item)
        if os.path.isdir(item_path):
            SAMPLE_FOLDER_OPTIONS.append(item)
    SAMPLE_FOLDER_OPTIONS.sort() # Sort for consistent display
    print(f'Found sample folders: {SAMPLE_FOLDER_OPTIONS}')
else:
    print(f'Error: Sample repository directory not found at {SAMPLE_REPO_DIR}. Please ensure the repository was cloned correctly.')

# --- Gradio UI for selection ---
# Global variable to store the selected sample folder
global SELECTED_SAMPLE_FOLDER
SELECTED_SAMPLE_FOLDER = None

def select_sample_folder(folder_name):
    global SELECTED_SAMPLE_FOLDER
    SELECTED_SAMPLE_FOLDER = folder_name
    return f"Selected sample folder: **{folder_name}**"

if SAMPLE_FOLDER_OPTIONS:
    with gr.Blocks(theme=gr.themes.Soft()) as sample_folder_selector_ui:
        gr.Markdown("## Select a Sample Folder for Analysis")
        gr.Markdown("Choose a sample project folder from the cloned repository. This selection will update the `SELECTED_SAMPLE_FOLDER` variable.")

        dropdown = gr.Dropdown(
            choices=SAMPLE_FOLDER_OPTIONS,
            label="Sample Project Folders",
            value=SAMPLE_FOLDER_OPTIONS[0] if SAMPLE_FOLDER_OPTIONS else None, # Pre-select first item if available
            interactive=True
        )
        output_text = gr.Markdown(value="Select a sample folder to begin.")

        dropdown.change(
            fn=select_sample_folder,
            inputs=[dropdown],
            outputs=[output_text]
        )

    print("\nLaunching Gradio UI for sample folder selection...")
    sample_folder_selector_ui.launch(share=True)

    # Set initial SELECTED_SAMPLE_FOLDER after launching the UI
    if SAMPLE_FOLDER_OPTIONS and SELECTED_SAMPLE_FOLDER is None:
        SELECTED_SAMPLE_FOLDER = SAMPLE_FOLDER_OPTIONS[0]
        print(f"Initial sample folder selection (if not changed in UI): {SELECTED_SAMPLE_FOLDER}")
else:
    print("No sample folders found to create a selection UI.")


# --- Functions for file analysis (inspired by user's prompt) ---

def create_threat_list_text() -> str:
    """Create a compact text representation of all EMB3D threats for prompting."""
    lines = []
    # Use global threats_list which is available from previous cells
    # Each item in threats_list has 'x_mitre_emb3d_threat_id' and 'name'
    for threat in threats_list:
        tid = threat.get('x_mitre_emb3d_threat_id', 'Unknown TID')
        name = threat.get('name', 'Unknown Threat Name')
        lines.append(f"- {tid}: {name}")
    return "\n".join(lines)

def fetch_mitigations_for_threat(threat_stix_id: str) -> list:
    """Retrieve mitigation descriptions from the EMB3D data for a given threat STIX ID."""
    mitigations_list = []
    if threat_stix_id in threat_to_coa_ids: # Use global threat_to_coa_ids
        coa_stix_ids = threat_to_coa_ids[threat_stix_id]
        for coa_id in coa_stix_ids:
            if coa_id in coa_by_id: # Use global coa_by_id
                desc = coa_by_id[coa_id].get("description", "No description available.")
                name = coa_by_id[coa_id].get("name", "Unknown Mitigation")
                mitigations_list.append({"name": name, "description": desc})
    return mitigations_list

def analyze_file_content_with_gemini(file_content: str, filename: str) -> dict:
    """Ask Gemini to find vulnerabilities and map to EMB3D threat IDs for generic file content."""
    threat_list_str = create_threat_list_text()
    prompt = f"""
    You are an embedded security expert using the EMB3D threat model.
    Below is a list of relevant EMB3D threats (ID and name only):
    {threat_list_str}

    Analyze the following file content for vulnerabilities that match one or more of these threats.
    For each vulnerability, output the most specific EMB3D threat ID (e.g., TID-101, TID-105, etc.)
    and a brief description. Consider the context of an embedded system.

    File: {filename}
    ```
    {file_content[:12000]}
    ```

    Return a JSON object with this structure:
    {{
      "vulnerabilities": [
        {{
          "tid": "TID-xxx",
          "description": "Short description (line if known)",
          "severity": "HIGH/MEDIUM/LOW"
        }}
      ],
      "risk_score": 0-10,
      "summary": "one sentence"
    }}

    ONLY return valid JSON.
    """
    try:
        # client and MODEL are expected to be globally available from previous cells
        response = client.models.generate_content(
            model=MODEL,
            contents=[genai_types.Part.from_text(text=prompt)],
            config=genai_types.GenerateContentConfig(
                max_output_tokens=8192,
                response_mime_type="application/json"
            )
        )
        text = response.text.strip().replace("```json", "").replace("```", "")
        return json.loads(text)
    except Exception as e:
        print(f"Parse or API call error for {filename}: {e}")
        return {"vulnerabilities": [], "risk_score": 0, "summary": f"Error during analysis: {e}"}

# --- New code to analyze selected folder files ---
analysis_results_per_file = [] # Global to store results for UI

if SELECTED_SAMPLE_FOLDER:
    full_sample_folder_path = os.path.join(SAMPLE_REPO_DIR, SELECTED_SAMPLE_FOLDER)
    print(f'\n--- Analyzing files in selected sample folder: {full_sample_folder_path} ---')

    combined_file_content_raw = [] # Store raw content as (filename, content) tuples
    for root, _, files in os.walk(full_sample_folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    # Store as a tuple (relative_filename, content) for easier processing
                    relative_file_path = os.path.relpath(file_path, full_sample_folder_path)
                    combined_file_content_raw.append((relative_file_path, content))
            except Exception as e:
                print(f"Could not read file {file_path}: {e}")

    if combined_file_content_raw:
        for filename, content in combined_file_content_raw:
            print(f"\nAnalyzing {filename}...")
            result = analyze_file_content_with_gemini(content, filename)
            analysis_results_per_file.append({"filename": filename, "analysis": result})
    else:
        print("No files found or readable in the selected folder for analysis.")

    # --- Gradio UI for File Analysis Results ---
    if analysis_results_per_file:
        file_choices = []
        for file_res in analysis_results_per_file:
            file_choices.append(file_res['filename'])
        file_choices.sort()

        def get_file_analysis_details(selected_filename):
            if not selected_filename:
                return "Select a file from the dropdown to view its analysis."

            for file_res in analysis_results_per_file:
                if file_res['filename'] == selected_filename:
                    analysis = file_res['analysis']
                    report_detail_str = f"## File: {selected_filename}\n\n"
                    report_detail_str += f"**Summary:** {analysis.get('summary', 'No summary available.')}\n"
                    report_detail_str += f"**Risk Score:** {analysis.get('risk_score', 'N/A')}\n\n"

                    if analysis.get("vulnerabilities"):
                        report_detail_str += "### Vulnerabilities:\n\n"
                        for vul in analysis["vulnerabilities"]:
                            threat_tid = vul.get("tid", "Unknown TID")
                            threat_desc = vul.get("description", "No description.")
                            severity = vul.get("severity", "N/A")
                            report_detail_str += f"<details><summary><b>- {threat_tid} (Severity: {severity}):</b> {threat_desc}</summary>\n"

                            # Attempt to fetch mitigations based on threat_tid
                            threat_stix_id_for_mitigation = None
                            for t in threats_list: # threats_list is global from EMB3D dataset loading
                                if t.get('x_mitre_emb3d_threat_id') == threat_tid:
                                    threat_stix_id_for_mitigation = t['id']
                                    break

                            if threat_stix_id_for_mitigation:
                                mitigations = fetch_mitigations_for_threat(threat_stix_id_for_mitigation)
                                if mitigations:
                                    report_detail_str += "  **Mitigations:**\n"
                                    for mit in mitigations:
                                        report_detail_str += f"    - **{mit['name']}**\n"
                                        report_detail_str += f"      Description: {textwrap.fill(mit['description'], width=70, initial_indent='        ', subsequent_indent='        ')}\n\n"
                                else:
                                    report_detail_str += "  *No specific mitigations found for this threat.*\n\n"
                            else:
                                report_detail_str += f"  *Could not find STIX ID for threat {threat_tid} to fetch mitigations.*\n\n"
                            report_detail_str += "</details>\n\n" # Close details tag
                    else:
                        report_detail_str += "  *No vulnerabilities identified.*\n"
                    return report_detail_str
            return "Analysis results for the selected file not found."

        def get_file_analysis_summary():
            total_files = len(analysis_results_per_file)
            total_vulnerabilities = 0
            unique_threat_ids = set()

            for file_res in analysis_results_per_file:
                analysis = file_res['analysis']
                if analysis.get("vulnerabilities"):
                    total_vulnerabilities += len(analysis["vulnerabilities"])
                    for vul in analysis["vulnerabilities"]:
                        unique_threat_ids.add(vul.get('tid', 'Unknown'))

            summary_str = f"**Total Files Analyzed:** {total_files}\n"
            summary_str += f"**Total Vulnerabilities Identified:** {total_vulnerabilities}\n"
            summary_str += f"**Total Unique Threats Identified:** {len(unique_threat_ids)}\n\n"
            summary_str += "Select a file from the dropdown below to view its detailed analysis."
            return summary_str

        with gr.Blocks(theme=gr.themes.Soft()) as file_analysis_ui:
            gr.Markdown(
                """
                # 🕵️‍♂️ EMB3D File Analysis Report
                ### Interactive Dashboard for Identified Threats and Mitigations in Sample Project Files
                """
            )

            gr.Markdown(get_file_analysis_summary())

            with gr.Row():
                with gr.Column(scale=1):
                    file_dropdown = gr.Dropdown(
                        choices=file_choices,
                        label="Select a File for Analysis",
                        value=None, # Initially no selection
                        interactive=True
                    )
                with gr.Column(scale=3):
                    output_markdown = gr.Markdown(
                        value="Select a file from the dropdown to view its details.",
                        label="File Details, Threats, and Mitigations"
                    )

            file_dropdown.change(
                fn=get_file_analysis_details,
                inputs=[file_dropdown],
                outputs=[output_markdown]
            )

        print("\nLaunching Gradio UI for file analysis results...")
        file_analysis_ui.launch(share=True)
    else:
        print('No analysis results generated for the sample project files.')
else:
    print("No sample folder selected for file analysis.")

Found sample folders: ['Automotive', 'FlightControls', 'Missiles', 'sample_vulnerable_C_project']


/tmp/ipykernel_797/916909335.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as sample_folder_selector_ui:



Launching Gradio UI for sample folder selection...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://51f1fd8b5ac12976b2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Initial sample folder selection (if not changed in UI): Automotive

--- Analyzing files in selected sample folder: SampleItems4SecurityAnalysis_repo/Automotive ---

Analyzing EngineControlUnit.h...
Parse or API call error for EngineControlUnit.h: Expecting property name enclosed in double quotes: line 8 column 5 (char 406)

Analyzing can_frame.h...
Parse or API call error for can_frame.h: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}

Analyzing EngineControl.cpp...


/tmp/ipykernel_797/916909335.py:234: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as file_analysis_ui:



Launching Gradio UI for file analysis results...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0754bb1dc1c70c1285.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
